# 02 — PubMed Evidence Exploration

**Stage 3 of the pipeline:** add **literature evidence** for each EGFR drug.

Flow: load drug recommendations (from notebook 01) → search PubMed per drug → fetch article summaries → save evidence + per-drug summary CSVs.

Uses **NCBI E-utilities** (`esearch` + `esummary`) with retries and a polite delay between requests.

### 1. Test notebook environment

In [1]:
import sys
import time
from pathlib import Path

import requests
import pandas as pd

print("Notebook is working")
print("Python executable:", sys.executable)

Notebook is working
Python executable: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/.venv/bin/python


### 2. Set project folders

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DIR)

Project root: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant
Processed data folder: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed


### 3. Load EGFR drug recommendations (from notebook 01)

In [3]:
recommendations_file = PROCESSED_DIR / "egfr_drug_recommendations.csv"

if not recommendations_file.exists():
    raise FileNotFoundError(
        "egfr_drug_recommendations.csv not found. Run notebook 01 first."
    )

drug_recommendations_df = pd.read_csv(recommendations_file)
print("Rows:", len(drug_recommendations_df))
drug_recommendations_df.head()

Rows: 76


,drug_name,molecule_chembl_id,action_type,mechanism_of_action,approval_status,max_phase,target_name
0,PANITUMUMAB,CHEMBL1201827,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
1,CETUXIMAB,CHEMBL1201577,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
2,ERLOTINIB HYDROCHLORIDE,CHEMBL1079742,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
3,GEFITINIB,CHEMBL939,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor
4,LAPATINIB DITOSYLATE,CHEMBL1201179,INHIBITOR,Epidermal growth factor receptor erbB1 inhibitor,Research / Unknown,4.0,Epidermal growth factor receptor


### 4. Select drugs to search
Start small (first 5) so we are polite to the NCBI servers. Increase later.

In [4]:
target_name = "EGFR"

if "drug_name" not in drug_recommendations_df.columns:
    raise ValueError("drug_name column not found in recommendations file.")

drugs_to_search = (
    drug_recommendations_df["drug_name"]
    .dropna()
    .drop_duplicates()
    .head(5)
    .tolist()
)

print("Target:", target_name)
print("Drugs selected for PubMed search:")
for drug in drugs_to_search:
    print("-", drug)

Target: EGFR
Drugs selected for PubMed search:
- PANITUMUMAB
- CETUXIMAB
- ERLOTINIB HYDROCHLORIDE
- GEFITINIB
- LAPATINIB DITOSYLATE


### 5. PubMed helper function (with retries)

In [5]:
PUBMED_SEARCH_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
PUBMED_SUMMARY_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi"


def pubmed_get(url, params, retries=3, pause=1):
    """GET JSON from PubMed / NCBI E-utilities with simple retries."""
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=(10, 60))
            if response.status_code == 200:
                return response.json()
            print(f"  attempt {attempt + 1}: HTTP {response.status_code}, retrying...")
        except requests.exceptions.RequestException as error:
            print(f"  attempt {attempt + 1}: {type(error).__name__}, retrying...")
        time.sleep(pause)
    return None


def build_query(drug):
    """A focused query: the drug AND EGFR, matched in title/abstract -> relevant results."""
    return f'("{drug}"[Title/Abstract]) AND (EGFR[Title/Abstract])'

### 6. Quick test: search PubMed for one drug

In [6]:
test_drug = drugs_to_search[0]
test_query = build_query(test_drug)

params = {"db": "pubmed", "term": test_query, "retmode": "json", "retmax": 5}
test_search_data = pubmed_get(PUBMED_SEARCH_URL, params)

if test_search_data is None:
    raise RuntimeError("PubMed test search failed.")

test_pmids = test_search_data["esearchresult"]["idlist"]
total = test_search_data["esearchresult"]["count"]
print("Query:", test_query)
print(f"Total matching articles: {total}  |  PMIDs fetched: {test_pmids}")

Query: ("PANITUMUMAB"[Title/Abstract]) AND (EGFR[Title/Abstract])
Total matching articles: 1202  |  PMIDs fetched: ['42307258', '42305981', '42295566', '42289363', '42276782']


### 7. Search PubMed for all selected drugs

In [7]:
pubmed_search_records = []

for drug in drugs_to_search:
    query = build_query(drug)
    params = {"db": "pubmed", "term": query, "retmode": "json", "retmax": 5}

    search_data = pubmed_get(PUBMED_SEARCH_URL, params)
    if search_data is None:
        print(f"Search failed for: {drug}")
        continue

    pmids = search_data["esearchresult"]["idlist"]
    total = int(search_data["esearchresult"]["count"])

    pubmed_search_records.append({
        "target_name": target_name,
        "drug_name": drug,
        "query": query,
        "pmids": pmids,
        "total_matches": total,
        "pubmed_count": len(pmids),
    })
    print(f"{drug}: {total} total matches, fetched {len(pmids)} PMIDs")
    time.sleep(0.4)  # be polite to NCBI

pubmed_search_df = pd.DataFrame(pubmed_search_records)
pubmed_search_df[["drug_name", "total_matches", "pubmed_count"]]

PANITUMUMAB: 1202 total matches, fetched 5 PMIDs
CETUXIMAB: 4006 total matches, fetched 5 PMIDs
ERLOTINIB HYDROCHLORIDE: 18 total matches, fetched 5 PMIDs
GEFITINIB: 5673 total matches, fetched 5 PMIDs
LAPATINIB DITOSYLATE: 5 total matches, fetched 5 PMIDs


,drug_name,total_matches,pubmed_count
0,PANITUMUMAB,1202,5
1,CETUXIMAB,4006,5
2,ERLOTINIB HYDROCHLORIDE,18,5
3,GEFITINIB,5673,5
4,LAPATINIB DITOSYLATE,5,5


### 8. Fetch article summaries (title, journal, date)

In [9]:
pubmed_article_records = []

for record in pubmed_search_records:
    drug = record["drug_name"]
    query = record["query"]
    pmids = record["pmids"]
    if not pmids:
        continue

    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "json"}
    summary_data = pubmed_get(PUBMED_SUMMARY_URL, params)
    if summary_data is None:
        print(f"Summary fetch failed for: {drug}")
        continue

    result_data = summary_data.get("result", {})
    for pmid in pmids:
        article = result_data.get(pmid, {})
        pubmed_article_records.append({
            "target_name": target_name,
            "drug_name": drug,
            "pmid": pmid,
            "title": article.get("title"),
            "journal": article.get("fulljournalname"),
            "publication_date": article.get("pubdate"),
            "query": query,
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
            "source": "PubMed",
        })
    print(f"Fetched summaries for: {drug}")
    time.sleep(0.4)

pubmed_evidence_df = pd.DataFrame(pubmed_article_records)
print("Total PubMed evidence records:", len(pubmed_evidence_df))
pubmed_evidence_df.head(10)

Fetched summaries for: PANITUMUMAB
Fetched summaries for: CETUXIMAB
Fetched summaries for: ERLOTINIB HYDROCHLORIDE
Fetched summaries for: GEFITINIB
Fetched summaries for: LAPATINIB DITOSYLATE
Total PubMed evidence records: 25


,target_name,drug_name,pmid,title,journal,publication_date,query,url,source
0,EGFR,PANITUMUMAB,42307258,Management of periungual pyogenic granulomas d...,Journal of the European Academy of Dermatology...,2026 Jun 17,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42307258/,PubMed
1,EGFR,PANITUMUMAB,42305981,Comparative efficacy and safety of first-line ...,Molecular and clinical oncology,2026 Aug,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42305981/,PubMed
2,EGFR,PANITUMUMAB,42295566,Panitumumab labeled with Auger electron-emitti...,EJNMMI radiopharmacy and chemistry,2026 Jun 15,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42295566/,PubMed
3,EGFR,PANITUMUMAB,42289363,Fluorescence-Guided Detection of Perineural an...,Head & neck,2026 Jun 14,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42289363/,PubMed
4,EGFR,PANITUMUMAB,42276782,Treatment of Human Glioblastoma Multiforme in ...,Journal of nuclear medicine : official publica...,2026 Jun 11,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42276782/,PubMed
5,EGFR,CETUXIMAB,42315817,MiR-31 as a predictive biomarker of cetuximab ...,Clinical & translational oncology : official p...,2026 Jun 18,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42315817/,PubMed
6,EGFR,CETUXIMAB,42314907,Overcoming Cetuximab Resistance in HNSCC by Hs...,The Journal of biological chemistry,2026 Jun 18,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42314907/,PubMed
7,EGFR,CETUXIMAB,42307258,Management of periungual pyogenic granulomas d...,Journal of the European Academy of Dermatology...,2026 Jun 17,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42307258/,PubMed
8,EGFR,CETUXIMAB,42305981,Comparative efficacy and safety of first-line ...,Molecular and clinical oncology,2026 Aug,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42305981/,PubMed
9,EGFR,CETUXIMAB,42284626,Intravenous amivantamab after cetuximab failur...,Oral oncology,2026 Jun 12,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42284626/,PubMed


### 9. Save the PubMed evidence dataset

In [10]:
pubmed_evidence_file = PROCESSED_DIR / "egfr_pubmed_evidence.csv"
pubmed_evidence_df.to_csv(pubmed_evidence_file, index=False)
print("Saved:", pubmed_evidence_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_pubmed_evidence.csv


### 10. Build a per-drug PubMed summary

In [11]:
if pubmed_evidence_df.empty:
    pubmed_summary_df = pd.DataFrame(
        columns=["target_name", "drug_name", "pubmed_count", "top_pubmed_titles"]
    )
else:
    pubmed_summary_df = (
        pubmed_evidence_df.groupby(["target_name", "drug_name"])
        .agg(
            pubmed_count=("pmid", "nunique"),
            top_pubmed_titles=("title", lambda s: " | ".join(list(s.dropna())[:3])),
        )
        .reset_index()
    )

pubmed_summary_df

,target_name,drug_name,pubmed_count,top_pubmed_titles
0,EGFR,CETUXIMAB,5,MiR-31 as a predictive biomarker of cetuximab ...
1,EGFR,ERLOTINIB HYDROCHLORIDE,5,Hypertrophic cardiomyopathy-associated mutatio...
2,EGFR,GEFITINIB,5,"Design, Synthesis of Novel Quinazolinone Deriv..."
3,EGFR,LAPATINIB DITOSYLATE,5,Unveiling the therapeutic prospects of EGFR in...
4,EGFR,PANITUMUMAB,5,Management of periungual pyogenic granulomas d...


### 11. Save the PubMed summary dataset

In [12]:
pubmed_summary_file = PROCESSED_DIR / "egfr_pubmed_summary.csv"
pubmed_summary_df.to_csv(pubmed_summary_file, index=False)
print("Saved:", pubmed_summary_file)

Saved: /Users/daniel/Documents/Projects/AI  Engineering Tools/datatalks/llm/ai-precision-medicine-lab/therapeutic-strategy-assistant/data/processed/egfr_pubmed_summary.csv


### 12. Final result

In [13]:
print("PubMed Evidence Exploration Complete")
print("=" * 60)
print("Target:", target_name)
print("Drugs searched:", len(drugs_to_search))
print("PubMed evidence records:", len(pubmed_evidence_df))
print("PubMed summary rows:", len(pubmed_summary_df))
display(pubmed_summary_df)
display(pubmed_evidence_df.head(10))

PubMed Evidence Exploration Complete
Target: EGFR
Drugs searched: 5
PubMed evidence records: 25
PubMed summary rows: 5


,target_name,drug_name,pubmed_count,top_pubmed_titles
0,EGFR,CETUXIMAB,5,MiR-31 as a predictive biomarker of cetuximab ...
1,EGFR,ERLOTINIB HYDROCHLORIDE,5,Hypertrophic cardiomyopathy-associated mutatio...
2,EGFR,GEFITINIB,5,"Design, Synthesis of Novel Quinazolinone Deriv..."
3,EGFR,LAPATINIB DITOSYLATE,5,Unveiling the therapeutic prospects of EGFR in...
4,EGFR,PANITUMUMAB,5,Management of periungual pyogenic granulomas d...


,target_name,drug_name,pmid,title,journal,publication_date,query,url,source
0,EGFR,PANITUMUMAB,42307258,Management of periungual pyogenic granulomas d...,Journal of the European Academy of Dermatology...,2026 Jun 17,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42307258/,PubMed
1,EGFR,PANITUMUMAB,42305981,Comparative efficacy and safety of first-line ...,Molecular and clinical oncology,2026 Aug,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42305981/,PubMed
2,EGFR,PANITUMUMAB,42295566,Panitumumab labeled with Auger electron-emitti...,EJNMMI radiopharmacy and chemistry,2026 Jun 15,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42295566/,PubMed
3,EGFR,PANITUMUMAB,42289363,Fluorescence-Guided Detection of Perineural an...,Head & neck,2026 Jun 14,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42289363/,PubMed
4,EGFR,PANITUMUMAB,42276782,Treatment of Human Glioblastoma Multiforme in ...,Journal of nuclear medicine : official publica...,2026 Jun 11,"(""PANITUMUMAB""[Title/Abstract]) AND (EGFR[Titl...",https://pubmed.ncbi.nlm.nih.gov/42276782/,PubMed
5,EGFR,CETUXIMAB,42315817,MiR-31 as a predictive biomarker of cetuximab ...,Clinical & translational oncology : official p...,2026 Jun 18,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42315817/,PubMed
6,EGFR,CETUXIMAB,42314907,Overcoming Cetuximab Resistance in HNSCC by Hs...,The Journal of biological chemistry,2026 Jun 18,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42314907/,PubMed
7,EGFR,CETUXIMAB,42307258,Management of periungual pyogenic granulomas d...,Journal of the European Academy of Dermatology...,2026 Jun 17,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42307258/,PubMed
8,EGFR,CETUXIMAB,42305981,Comparative efficacy and safety of first-line ...,Molecular and clinical oncology,2026 Aug,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42305981/,PubMed
9,EGFR,CETUXIMAB,42284626,Intravenous amivantamab after cetuximab failur...,Oral oncology,2026 Jun 12,"(""CETUXIMAB""[Title/Abstract]) AND (EGFR[Title/...",https://pubmed.ncbi.nlm.nih.gov/42284626/,PubMed
